# 04B 真实 CIF → descriptors → ML table

> 🔵 **Level B · 建议掌握** | 科研实践 | 完成标准：建立可复现的 `CIF → descriptor row → COF_ID → target row` 映射。

向前追溯一步：`real COF CIF → parse → QC → descriptors → COF_ID → target table → ML`。使用 CURATED-COFs 的真实结构。

In [ ]:
!pip -q install pymatgen requests
import pandas as pd, requests
from pymatgen.core import Structure
meta=pd.read_csv('https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cof-frameworks.csv').rename(columns={'CURATED-COFs ID':'COF_ID'})
display(meta.head())


In [ ]:
N_COF=50
rows=[]; failed=[]
for cof_id in meta['COF_ID'].dropna().astype(str).head(N_COF):
    url=f'https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cifs/{cof_id}.cif'
    try:
        s=Structure.from_str(requests.get(url,timeout=20).text,fmt='cif'); comp=s.composition.fractional_composition
        row={'COF_ID':cof_id,'a_A':s.lattice.a,'b_A':s.lattice.b,'c_A':s.lattice.c,'alpha':s.lattice.alpha,'beta':s.lattice.beta,'gamma':s.lattice.gamma,'volume_A3':s.volume,'density_g_cm3':float(s.density),'n_atoms':len(s),'n_elements':len(s.composition.elements)}
        for el in ['H','B','C','N','O','F','S']: row[f'{el}_fraction']=float(comp.get_atomic_fraction(el))
        rows.append(row)
    except Exception as e: failed.append((cof_id,str(e)))
features=pd.DataFrame(rows)
print('parsed =',len(features),'failed =',len(failed)); display(features.head())


COF_ID 是结构、pore descriptors 和 target tables 的主键。PLD/LCD/ASA/void fraction/pore volume 需要专门 pore analysis，不能把晶胞尺寸直接当孔径。
